In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_1.py ──
"""
Shared infrastructure for MLFP02 Exercise 1 — Probability and Bayesian
Fundamentals.

Contains: HDB data loading (4-ROOM 2020+ slice), prior/posterior math for
the Normal-Normal and Beta-Binomial conjugate families, bootstrap helpers,
and output directory wiring.

Technique-specific narrative (MLE derivation, Bayes scenarios, credible vs
confidence simulation, bootstrap comparison) does NOT belong here — it
lives in the per-technique files.
"""

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import polars as pl
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

# Output directory for plots (HTML) — every technique writes here.
OUTPUT_DIR = Path("outputs") / "mlfp02_ex1_bayes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Canonical prior used across all four technique files. A single source
# of truth means Task 5's prior sweep and Task 4's posterior use the same
# starting point.
PRIOR_MU_0: float = 500_000.0  # SGD — Singapore 4-room HDB market anchor
PRIOR_SIGMA_0: float = 100_000.0  # SGD — moderate uncertainty

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale flats (Singapore, data.gov.sg)
# ════════════════════════════════════════════════════════════════════════

_loader = MLFPDataLoader()


def load_hdb_all() -> pl.DataFrame:
    """Full HDB resale dataset filtered to 2020+ transactions.

    Returns a polars DataFrame with columns: month, town, flat_type,
    resale_price, plus any others present in the source parquet. Used
    by Task 1 (truth tables) and Task 8 (expected value by flat type).
    """
    hdb = _loader.load("mlfp01", "hdb_resale.parquet")
    return hdb.filter(pl.col("month").str.to_date("%Y-%m") >= pl.date(2020, 1, 1))


def load_hdb_4room() -> pl.DataFrame:
    """4-ROOM slice of HDB resale (2020+) — the primary estimation target."""
    return load_hdb_all().filter(pl.col("flat_type") == "4 ROOM")


def load_hdb_prices_4room() -> np.ndarray:
    """Return the 4-room resale_price column as a float64 numpy array.

    This is the primary observation vector for MLE, Normal-Normal, and
    bootstrap tasks.
    """
    return load_hdb_4room()["resale_price"].to_numpy().astype(np.float64)


# ════════════════════════════════════════════════════════════════════════
# NORMAL-NORMAL CONJUGATE
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class NormalPosterior:
    """Posterior for μ under a Normal-Normal conjugate with known σ."""

    mean: float
    std: float
    prior_mean: float
    prior_std: float
    n: int
    sigma_known: float

    @property
    def precision_prior(self) -> float:
        return 1.0 / self.prior_std**2

    @property
    def precision_data(self) -> float:
        return self.n / self.sigma_known**2

    @property
    def prior_weight(self) -> float:
        """Fraction of posterior precision contributed by the prior (0..1)."""
        return self.precision_prior / (self.precision_prior + self.precision_data)

    def credible_interval(self, level: float = 0.95) -> tuple[float, float]:
        z = stats.norm.ppf(0.5 + level / 2)
        return (self.mean - z * self.std, self.mean + z * self.std)


def normal_normal_posterior(
    data: np.ndarray,
    prior_mean: float,
    prior_std: float,
    sigma_known: float,
) -> NormalPosterior:
    """Closed-form posterior for μ under N(μ₀, σ₀²) prior and known σ.

    Posterior precision = prior precision + n / σ². Posterior mean is the
    precision-weighted average of the prior mean and the sample mean.
    """
    n = len(data)
    xbar = float(np.mean(data))
    prec_prior = 1.0 / prior_std**2
    prec_data = n / sigma_known**2
    prec_post = prec_prior + prec_data
    sigma_post_sq = 1.0 / prec_post
    mu_post = sigma_post_sq * (prior_mean * prec_prior + n * xbar / sigma_known**2)
    return NormalPosterior(
        mean=mu_post,
        std=float(np.sqrt(sigma_post_sq)),
        prior_mean=prior_mean,
        prior_std=prior_std,
        n=n,
        sigma_known=sigma_known,
    )


# ════════════════════════════════════════════════════════════════════════
# BETA-BINOMIAL CONJUGATE
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class BetaPosterior:
    """Posterior for p under a Beta-Binomial conjugate."""

    alpha: float
    beta: float
    prior_alpha: float
    prior_beta: float
    k: int
    n: int

    @property
    def mean(self) -> float:
        return self.alpha / (self.alpha + self.beta)

    @property
    def prior_mean(self) -> float:
        return self.prior_alpha / (self.prior_alpha + self.prior_beta)

    def credible_interval(self, level: float = 0.95) -> tuple[float, float]:
        lo = (1 - level) / 2
        hi = 1 - lo
        return tuple(stats.beta.ppf([lo, hi], self.alpha, self.beta).tolist())


def beta_binomial_posterior(
    k: int, n: int, prior_alpha: float, prior_beta: float
) -> BetaPosterior:
    """Closed-form posterior for p under Beta(α, β) prior and k/n Binomial."""
    return BetaPosterior(
        alpha=prior_alpha + k,
        beta=prior_beta + (n - k),
        prior_alpha=prior_alpha,
        prior_beta=prior_beta,
        k=k,
        n=n,
    )


# ════════════════════════════════════════════════════════════════════════
# MLE + CRAMER-RAO
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class NormalMLE:
    mean: float
    mle_std: float  # ddof=0, biased
    unbiased_std: float  # ddof=1, Bessel's correction
    n: int

    @property
    def fisher_information(self) -> float:
        return self.n / self.mle_std**2

    @property
    def cramer_rao_bound(self) -> float:
        return 1.0 / self.fisher_information

    @property
    def standard_error(self) -> float:
        return float(np.sqrt(self.cramer_rao_bound))


def normal_mle(data: np.ndarray) -> NormalMLE:
    """MLE for Normal (μ, σ²) with Cramér-Rao bound bookkeeping."""
    arr = np.asarray(data, dtype=np.float64)
    return NormalMLE(
        mean=float(arr.mean()),
        mle_std=float(arr.std(ddof=0)),
        unbiased_std=float(arr.std(ddof=1)),
        n=len(arr),
    )


# ════════════════════════════════════════════════════════════════════════
# BOOTSTRAP INTERVALS
# ════════════════════════════════════════════════════════════════════════


def bootstrap_mean_distribution(
    data: np.ndarray, n_bootstrap: int = 10_000, seed: int = 42
) -> np.ndarray:
    """Non-parametric bootstrap of the sample mean."""
    rng = np.random.default_rng(seed)
    n = len(data)
    return np.array(
        [rng.choice(data, size=n, replace=True).mean() for _ in range(n_bootstrap)]
    )


def percentile_ci(draws: np.ndarray, level: float = 0.95) -> tuple[float, float]:
    lo = (1 - level) / 2 * 100
    hi = (1 + level) / 2 * 100
    return float(np.percentile(draws, lo)), float(np.percentile(draws, hi))


def bca_ci(
    data: np.ndarray, n_bootstrap: int = 10_000, seed: int = 42, level: float = 0.95
) -> tuple[float, float]:
    """Bias-corrected accelerated bootstrap CI for the mean (via scipy)."""
    result = stats.bootstrap(
        (np.asarray(data, dtype=np.float64),),
        statistic=np.mean,
        n_resamples=n_bootstrap,
        confidence_level=level,
        method="BCa",
        random_state=seed,
    )
    return float(result.confidence_interval.low), float(result.confidence_interval.high)


# ════════════════════════════════════════════════════════════════════════
# FORMATTING
# ════════════════════════════════════════════════════════════════════════


def fmt_money(x: float) -> str:
    return f"${x:,.0f}"


def print_interval(label: str, lo: float, hi: float) -> None:
    print(
        f"  {label:<28} [{fmt_money(lo)}, {fmt_money(hi)}]  width={fmt_money(hi - lo)}"
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 1.1: Probability Fundamentals and Maximum Likelihood
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Construct truth tables for probability problems and compute joint /
#     conditional probabilities from real HDB transaction data
#   - Test whether two events are independent using the product rule
#   - Compute MLE for Normal distribution parameters (μ̂, σ̂²)
#   - Quantify estimation uncertainty via the Cramér-Rao lower bound
#   - Apply MLE precision to a Singapore property valuation scenario
#
# PREREQUISITES: Complete M1 — comfortable loading data, computing
#   summary statistics, and reading Polars DataFrames.
#
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Theory — what probability means for property valuation
#   2. Build — load HDB data, compute truth table and joint probs
#   3. Train — MLE for Normal parameters with Cramér-Rao bound
#   4. Visualise — cross-tabulation heatmap + MLE precision plot
#   5. Apply — Singapore property valuation: how precise is "the average"?
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats



## THEORY — What Probability Means for Property Valuation


In [ ]:
# Probability begins with counting. Before any Bayesian update, before any
# machine learning model, we need to know how to answer:
#
#   "What fraction of events satisfy condition A AND condition B?"
#   "Given that A occurred, what is the probability of B?"
#
# These are joint and conditional probabilities — the building blocks of
# every statistical model. In property valuation, this translates to
# questions like:
#
#   "If I know a flat is 4-room, what's the chance it sold above $500K?"
#   "Are flat type and price range independent, or does one tell me
#    something about the other?"
#
# Key rules:
#   P(A) + P(A') = 1
#   P(A,B) = P(A) × P(B|A)
#   Independent events: P(A,B) = P(A) × P(B)
#
# Maximum Likelihood Estimation (MLE) extends counting to continuous
# distributions. For X ~ N(μ, σ²):
#   μ̂ = x̄   (sample mean)
#   σ̂² = (1/n) Σ(xᵢ - x̄)²   (biased variance, ddof=0)
#
# The Fisher information I(μ) = n/σ² tells us how much the data reveals
# about μ. The Cramér-Rao bound says Var(μ̂) ≥ 1/I(μ) = σ²/n — no
# unbiased estimator can do better.



## TASK 2 — BUILD: Load HDB data and compute truth table


In [ ]:
print("\n" + "=" * 70)
print("  MLFP02 Exercise 1.1: Probability Fundamentals & MLE")
print("=" * 70)

# Load data via shared helpers
prices = load_hdb_prices_4room()
hdb_all = load_hdb_all()
total_n = hdb_all.height

print(f"\n  Data loaded: {len(prices):,} 4-room HDB transactions (2020+)")
print(f"  Price range: {fmt_money(prices.min())} – {fmt_money(prices.max())}")
print(f"  Sample mean: {fmt_money(prices.mean())}")
print(f"  Sample std:  {fmt_money(prices.std())}\n")



### Event definitions


In [ ]:
# Event A: transaction is a 4-room flat
# TODO: Filter hdb_all to 4 ROOM and get the count
# Hint: hdb_all.filter(pl.col("flat_type") == "____").height
n_4room = ____
p_4room = n_4room / total_n

# Event B: price above $500K
# TODO: Filter hdb_all for resale_price > 500_000 and get count
n_above_500k = ____
p_above_500k = n_above_500k / total_n

# Joint probability: P(4-room AND price > 500K)
# TODO: Filter with BOTH conditions using & and compute the joint count
n_4room_and_above = ____
p_joint = n_4room_and_above / total_n

# Conditional: P(price > 500K | 4-room) = P(A,B) / P(A)
# TODO: Compute the conditional probability
# Hint: divide the joint count by the 4-room count
p_above_given_4room = ____

# Test independence: P(A,B) vs P(A)×P(B)
p_independent = p_4room * p_above_500k

print("--- Truth Table (empirical) ---")
print(f"Total transactions (2020+): {total_n:,}")
print(f"P(4-room)           = {p_4room:.4f} ({p_4room:.1%})")
print(f"P(price > $500K)    = {p_above_500k:.4f} ({p_above_500k:.1%})")
print(f"P(4-room AND >$500K)= {p_joint:.4f} ({p_joint:.1%})")
print(f"P(>$500K | 4-room)  = {p_above_given_4room:.4f} ({p_above_given_4room:.1%})")
print(f"\n--- Independence Check ---")
print(f"P(A)×P(B) = {p_independent:.4f}")
print(f"P(A,B)    = {p_joint:.4f}")
print(f"Difference: {abs(p_joint - p_independent):.4f}")
if abs(p_joint - p_independent) < 0.01:
    print("Events are approximately independent")
else:
    print("Events are NOT independent — flat type affects price probability")
# INTERPRETATION: If flat type and price are not independent, knowing
# the flat type tells you something about the price distribution. This
# is the foundation for conditional reasoning in property valuation.

# Cross-tabulation: flat_type × price_category
# TODO: Create price bands using pl.when().then().otherwise()
# Hint: ≤400K, 400K-600K, 600K-800K, >800K
price_cats = hdb_all.with_columns(
    pl.when(pl.col("resale_price") <= 400_000)
    .then(pl.lit("≤400K"))
    .when(pl.col("resale_price") <= 600_000)
    .then(pl.lit("400K-600K"))
    .when(pl.col("resale_price") <= 800_000)
    .then(pl.lit("600K-800K"))
    .otherwise(pl.lit(">800K"))
    .alias("price_band")
)
cross_tab = (
    price_cats.group_by("flat_type", "price_band")
    .agg(pl.len().alias("count"))
    .sort("flat_type", "price_band")
)
print(f"\n--- Cross-Tabulation (sample) ---")
print(cross_tab.head(12))



### Checkpoint 1


In [ ]:
assert 0 < p_4room < 1, "P(4-room) must be a valid probability"
assert 0 < p_above_500k < 1, "P(>500K) must be a valid probability"
assert p_joint <= min(p_4room, p_above_500k), "Joint prob cannot exceed marginals"
assert (
    abs(p_above_given_4room - p_joint / p_4room) < 1e-10
), "Conditional probability identity must hold: P(B|A) = P(A,B)/P(A)"
print("\n✓ Checkpoint 1 passed — probability fundamentals computed\n")



## TASK 3 — TRAIN: Maximum Likelihood Estimation with Cramér-Rao bound


In [ ]:
# TODO: Compute MLE using the shared helper
# Hint: normal_mle(prices) returns a NormalMLE dataclass
mle = ____

print("=== MLE Estimates ===")
print(f"μ̂ = {fmt_money(mle.mean)}")
print(f"σ̂ (MLE, ddof=0)     = {fmt_money(mle.mle_std)}")
print(f"σ̂ (unbiased, ddof=1) = {fmt_money(mle.unbiased_std)}")
print(f"Bias: MLE σ underestimates by ${mle.unbiased_std - mle.mle_std:,.2f}")
print(f"\nFisher information I(μ) = {mle.fisher_information:.4f}")
print(f"Cramér-Rao lower bound: Var(μ̂) ≥ {mle.cramer_rao_bound:.2f}")
print(f"MLE standard error: ${mle.standard_error:,.2f}")
# INTERPRETATION: The standard error tells you the precision of the
# mean estimate. With many thousands of transactions, the SE is tiny
# relative to the mean — our estimate of the average 4-room HDB price
# is very precise. The Cramér-Rao bound guarantees no unbiased estimator
# can do better than this SE for the Normal model.



### Checkpoint 2


In [ ]:
assert mle.n > 0, "No data loaded — check the filter conditions"
assert mle.mean > 0, "MLE mean should be positive (price cannot be zero)"
assert mle.mle_std > 0, "MLE std should be positive"
assert mle.standard_error > 0, "Standard error should be positive"
assert mle.standard_error < mle.mle_std, "SE of mean should be much smaller than std"
assert (
    mle.unbiased_std > mle.mle_std
), "Unbiased σ must be > MLE σ (Bessel's correction)"
print("\n✓ Checkpoint 2 passed — MLE estimates and Cramér-Rao bound computed\n")



## TASK 4 — VISUALISE: Cross-tabulation heatmap + MLE precision


In [ ]:
# -- Plot 1: Cross-tab heatmap of flat_type × price_band --
pivot = cross_tab.pivot(on="price_band", index="flat_type", values="count").fill_null(0)
flat_types_order = sorted(pivot["flat_type"].to_list())
bands = ["≤400K", "400K-600K", "600K-800K", ">800K"]
z_matrix = []
for ft in flat_types_order:
    row_data = pivot.filter(pl.col("flat_type") == ft)
    row = [int(row_data[b].item()) if b in row_data.columns else 0 for b in bands]
    z_matrix.append(row)

# TODO: Create a heatmap using go.Heatmap with z=z_matrix, x=bands, y=flat_types_order
# Hint: go.Figure(data=go.Heatmap(z=____, x=____, y=____, colorscale="Blues"))
fig1 = ____
fig1.update_layout(
    title="HDB Cross-Tabulation: Flat Type × Price Band (2020+)",
    xaxis_title="Price Band",
    yaxis_title="Flat Type",
    height=400,
)
fig1.write_html(str(OUTPUT_DIR / "cross_tab_heatmap.html"))
print("Saved: cross_tab_heatmap.html")

# -- Plot 2: MLE precision — sample size vs standard error --
sample_sizes = np.arange(10, mle.n + 1, max(1, mle.n // 200))
# TODO: Compute SE for each sample size using the formula SE = σ / √n
# Hint: mle.mle_std / np.sqrt(sample_sizes)
se_values = ____

fig2 = go.Figure()
fig2.add_trace(
    go.Scatter(
        x=sample_sizes,
        y=se_values,
        mode="lines",
        name="SE = σ / √n",
        line={"color": "blue"},
    )
)
fig2.add_hline(
    y=mle.standard_error,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Actual SE = ${mle.standard_error:,.0f}",
)
fig2.update_layout(
    title="MLE Precision: How Standard Error Shrinks with Sample Size",
    xaxis_title="Sample Size (n)",
    yaxis_title="Standard Error ($)",
    height=400,
)
fig2.write_html(str(OUTPUT_DIR / "mle_precision.html"))
print("Saved: mle_precision.html")



### Checkpoint 3


In [ ]:
assert len(z_matrix) > 0, "Cross-tab matrix should not be empty"
print("\n✓ Checkpoint 3 passed — visualisations saved\n")



## TASK 5 — APPLY: Singapore Property Valuation — How Precise Is
         "The Average"?


In [ ]:
# A property developer planning a 4-room HDB launch in Queenstown wants
# to know the average resale price to set a competitive listing. They
# pull data from data.gov.sg.
#
# With n transactions, MLE gives μ̂ ± SE. The developer's margin is $20K
# per unit. If SE > $20K, the estimate is not precise enough to make a
# confident pricing decision — they need more data or a tighter segment.

print("=== APPLICATION: Property Developer Pricing Decision ===")
margin = 20_000
print(f"\nDeveloper margin per unit: {fmt_money(margin)}")
print(f"MLE estimate: {fmt_money(mle.mean)} ± {fmt_money(mle.standard_error)}")

# TODO: Compare mle.standard_error with margin and print the decision
# Hint: if mle.standard_error < margin → precise enough
if mle.standard_error < margin:
    print(f"\n✓ SE ({fmt_money(mle.standard_error)}) < margin ({fmt_money(margin)})")
    print("  → The estimate is precise enough for confident pricing.")
    print(f"  → List at {fmt_money(mle.mean)} with {fmt_money(margin)} margin")
    print(
        f"    gives a range of {fmt_money(mle.mean - margin)} – {fmt_money(mle.mean + margin)}"
    )
else:
    print(f"\n✗ SE ({fmt_money(mle.standard_error)}) ≥ margin ({fmt_money(margin)})")
    print("  → Need a narrower segment or more data for confident pricing.")

# TODO: Compute n_needed for SE < $5K using n ≥ (σ / target_SE)²
# Hint: int(np.ceil((mle.mle_std / target_se) ** 2))
target_se = 5_000
n_needed = ____
print(f"\nTo achieve SE < {fmt_money(target_se)}: need n ≥ {n_needed:,} transactions")
print(
    f"  (currently have {mle.n:,} — {'sufficient' if mle.n >= n_needed else 'insufficient'})"
)



### Checkpoint 4


In [ ]:
assert margin > 0, "Margin must be positive"
assert n_needed > 0, "Required n must be positive"
print("\n✓ Checkpoint 4 passed — business application complete\n")



## REFLECTION


✓ Joint probability P(A,B) and conditional P(A|B) from real data
  ✓ Independence test: P(A,B) vs P(A)×P(B) — flat type and price
    are NOT independent
  ✓ Cross-tabulation reveals where the market volume concentrates
  ✓ MLE for Normal: μ̂ = x̄, σ̂² with ddof=0 (biased) vs ddof=1
  ✓ Cramér-Rao bound: MLE achieves minimum possible variance
  ✓ Business framing: SE determines whether the estimate is
    actionable for a $20K pricing margin

  NEXT: In 02_bayes_theorem.py, you'll apply Bayes' theorem to
  medical testing and property valuation — learning why base rates
  make positive tests less trustworthy than you'd expect.


In [ ]:
print("═" * 70)
print("  WHAT YOU'VE MASTERED (1.1 — Probability & MLE)")
print("═" * 70)
print(
)

print("\n✓ Exercise 1.1 complete — Probability Fundamentals & MLE")

